# Historical collection of Twitch moderation records

This notebook documents the OAuth and Twitch API collection used before analysis. It fetches ban records and the archived categories of unban requests. Results contain personal and moderation information and are stored under the private collection directory.

Dependencies are `requests`, `pandas` and `openpyxl`. Supply `TWITCH_CLIENT_ID`, `TWITCH_CLIENT_SECRET`, `TWITCH_REDIRECT_URI`, `TWITCH_BROADCASTER_ID` and `TWITCH_AUTHORIZATION_CODE` as local environment variables for a broadcaster whose records you are authorized to access. The authorization code is used for a single collection call.

Running all cells defines the collection and export functions. The live call is commented out. This archived collection was kept separate from the course analysis rerun.


In [ ]:
import os
from pathlib import Path
import tempfile

repo_root = Path.cwd()
if repo_root.name == "collection" and repo_root.parent.name == "notebooks":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

private_data = Path(
    os.getenv("TWITCH_MODERATION_DATA") or repo_root / "data" / "private"
).expanduser()
private_collection = private_data / "collection"

import pandas as pd
import requests


In [ ]:
def fetch_twitch_data(broadcaster_id, code):
    client_id = os.environ["TWITCH_CLIENT_ID"]
    client_secret = os.environ["TWITCH_CLIENT_SECRET"]
    redirect_uri = os.environ["TWITCH_REDIRECT_URI"]
    token_url = "https://id.twitch.tv/oauth2/token"
    # Data for the POST request to obtain the access token
    payload = {
        'client_id': client_id,
        'client_secret': client_secret,
        'code': code,
        'grant_type': 'authorization_code',
        'redirect_uri': redirect_uri
    }

    # Send the POST request
    response = requests.post(token_url, data=payload)

    if response.status_code == 200:
        # The response includes the access token
        data = response.json()
        access_token = data['access_token']
        
    else:
        print(f"Error obtaining token: {response.status_code}")
        raise RuntimeError(f"Twitch authorization failed with status {response.status_code}.")

    headers = {
        'Client-Id': client_id,
        'Authorization': 'Bearer ' + access_token
    }

    # Initialize an empty dataframe for banned users
    df_banned = pd.DataFrame()

    # Base URL for the API request
    base_url = 'https://api.twitch.tv/helix/moderation/banned'
    url = f'{base_url}?broadcaster_id={broadcaster_id}'

    while url:
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            data = response.json()['data']
            cursor = response.json()['pagination'].get('cursor', None)
            
            # Add the data to the dataframe
            df_banned = pd.concat([df_banned, pd.DataFrame(data)], ignore_index=True)

            if cursor:
                # Update the URL to include the cursor for the next page of data
                url = f'{base_url}?broadcaster_id={broadcaster_id}&after={cursor}'
            else:
                # No more cursors, exit the loop
                url = None
        else:
            print(f'Error in banned users request: {response.status_code}')
            break

    # Initialize an empty dataframe for unban requests
    df_banned_req = pd.DataFrame()

    # Base URL for the API request
    base_url = 'https://api.twitch.tv/helix/moderation/unban_requests'
    statuses = ["pending", "approved", "denied", "acknowledged", "canceled"]

    for status in statuses:
        url = f"{base_url}?broadcaster_id={broadcaster_id}&moderator_id={broadcaster_id}&status={status}"

        while url:
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                json_response = response.json()
                data = json_response.get('data', [])
                cursor = json_response.get('pagination', {}).get('cursor')

                # Add the data to the dataframe
                df_banned_req = pd.concat([df_banned_req, pd.DataFrame(data)], ignore_index=True)

                if cursor:
                    url = f"{base_url}?broadcaster_id={broadcaster_id}&moderator_id={broadcaster_id}&status={status}&after={cursor}"
                else:
                    url = None
            else:
                print(f'Error in unban requests: {response.status_code}')
                break

    return df_banned, df_banned_req


In [ ]:
def export_moderation_data():
    broadcaster_id = os.environ["TWITCH_BROADCASTER_ID"]
    authorization_code = os.environ["TWITCH_AUTHORIZATION_CODE"]
    bans, appeals = fetch_twitch_data(broadcaster_id, authorization_code)

    private_collection.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="moderation-", dir=private_collection))
    bans.to_excel(run_dir / "bans.xlsx", index=False)
    appeals.to_excel(run_dir / "unban_requests.xlsx", index=False)


# Run this explicitly only for a channel you are authorized to access.
# export_moderation_data()
